# Liu2024 Source MAT EEGNetv4 Baseline

This notebook is a separate deep-learning baseline for the Liu2024 acute-stroke motor imagery dataset.

It reuses the source `.mat` loading logic from the S-JEPA notebook, but it does **not** force S-JEPA token/window constraints. The default path is EEGNet-oriented:

- select the 29 EEG channels used in the Liu paper setup, dropping CPz reference, EOG, and marker channels;
- average-reference the EEG;
- bandpass filter for broad MI decoding;
- resample to 128 Hz;
- crop a standard 4.0-second MI window;
- train Braindecode `EEGNetv4` within each subject;
- standardize using training-fold statistics only before testing.

The goal is a clean, reportable deep baseline, not a new S-JEPA variant.

# 1. Setup

In [ ]:
import os
import re
import sys
import json
import math
import hashlib
import random
import builtins
import platform
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset

from scipy.io import loadmat

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

from skorch.callbacks import EarlyStopping
from skorch.dataset import ValidSplit

from braindecode import EEGClassifier
from braindecode.models import EEGNetv4

import mne

mne.set_log_level("WARNING")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


In [ ]:
print("Runtime Environment:")
print(f"  - Python: {sys.version}")
print(f"  - Platform: {platform.platform()}")

WORKING_DIR = Path.cwd().resolve().parent.parent
print(f"\nWorking directory: {WORKING_DIR}")


# 2. Configuration

## 2.1 Liu2024 Channel Defaults

In [ ]:
# Liu2024 source MAT channel conventions.
# Source files are organized as trials x 33 channels x samples:
#   0..29 = EEG-like channels, index 17 = CPz source reference,
#   30..31 = EOG, 32 = marker.
#
# The channel labels below follow the Liu2024 paper / EEGLAB location files.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]


## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # Paths
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-eegnetv4"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),

    # Dataset
    "subjects_to_use": None,
    "exclude_subjects": [],

    # EEGNet-oriented preprocessing
    # 4-40 Hz is a broad MI/deep-learning band. Change to 8-30 if you want a Liu classical-band variant.
    "sfreq": 128,
    "bandpass_low": 4.0,
    "bandpass_high": 40.0,
    "apply_average_reference": True,
    "baseline_correct_epochs": True,

    # EEGNet windowing. This is deliberately not the S-JEPA 537-sample token-compatible window.
    "mi_window_start_s": 0.0,
    "mi_window_duration_s": 4.0,

    # Fold-level input normalization. This is done after the train/test split to avoid test leakage.
    "standardize_using_train_fold": True,
    "standardize_mode": "channel",  # channel = one mean/std per channel from training fold

    # Model
    "model_name": "EEGNetv4",
    "eegnet_F1": 8,
    "eegnet_D": 2,
    "eegnet_F2": 16,
    "eegnet_kernel_length": 64,
    "eegnet_drop_prob": 0.50,
    "eegnet_pool_mode": "mean",

    # Cross-validation
    "cv_folds": 5,
    "val_split": 0.25,
    "assert_balanced_folds": True,

    # Training
    "batch_size": 16,
    "n_epochs": 300,
    "early_stopping_patience": 40,
    "learning_rate": 1e-3,
    "weight_decay": 0.0,

    # Reproducibility
    "seed": 12,
    "set_seed": True,

    # Diagnostics
    "collapse_threshold": 0.90,
    "log_probability_diagnostics": True,
}


In [ ]:
# Liu2024 source MAT constants.
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
LIU_EXPECTED_SOURCE_CHANNELS = 33
LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL = 4000

# Liu source MAT files include EEG + EOG + marker channels. We keep the 29 EEG channels
# used in the Liu paper baseline and drop CPz because it is the source reference channel.
EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [
    name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
    if idx != SOURCE_REFERENCE_INDEX
]

SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])

TARGET_N_CLASSES = 2
WINDOW_SAMPLES = int(round(float(CONFIG["mi_window_duration_s"]) * float(CONFIG["sfreq"])))
TARGET_TRIAL_DURATION_S = WINDOW_SAMPLES / float(CONFIG["sfreq"])
MI_WINDOW_START_SAMPLE = int(round(float(CONFIG["mi_window_start_s"]) * float(CONFIG["sfreq"])))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + WINDOW_SAMPLES

print("Effective Liu2024 EEGNetv4 settings:")
print(f"  Channels:                {len(EEG_CHANNEL_NAMES)}")
print(f"  Channel names:           {EEG_CHANNEL_NAMES}")
print(f"  Source sfreq:            {LIU_SOURCE_SFREQ} Hz")
print(f"  Target sfreq:            {CONFIG['sfreq']} Hz")
print(f"  Bandpass:                {CONFIG['bandpass_low']}–{CONFIG['bandpass_high']} Hz")
print(f"  Average reference:       {CONFIG['apply_average_reference']}")
print(f"  Epoch baseline correct:  {CONFIG['baseline_correct_epochs']}")
print(f"  MI window start:         {CONFIG['mi_window_start_s']} s")
print(f"  Target window:           {TARGET_TRIAL_DURATION_S:.4f} s / {WINDOW_SAMPLES} samples")
print(f"  Train-fold standardize:  {CONFIG['standardize_using_train_fold']} ({CONFIG['standardize_mode']})")


## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

_ORIGINAL_PRINT = builtins.print

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass

    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")


## 2.4 Reproducibility

In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"]) if CONFIG["seed"] is not None else None
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED) # type: ignore[arg-type]
    print(f"Seed initialized: {BASE_SEED}")
else:
    print("Seed was not fixed because CONFIG['set_seed'] is False.")


# 3. Load and Prepare Data

## 3.1 Data Loading Helpers

In [ ]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    """Recursively walk scipy-loaded MATLAB dicts/structs.

    Liu2024 source files may expose only a top-level `eeg` object instead of
    top-level `rawdata` and `labels`. This walker lets the loader find nested
    arrays without assuming one exact MATLAB struct layout.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def mat_structure_preview(path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({
                "name": name,
                "type": "ndarray",
                "shape": str(value.shape),
                "dtype": str(value.dtype),
            })
        else:
            rows.append({
                "name": name,
                "type": type(value).__name__,
                "shape": "",
                "dtype": "",
            })
    return pd.DataFrame(rows).head(max_rows)

def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    # Prefer the label-count axis as the trial axis when labels are available.
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # After trial-axis normalization, the time axis should be the largest axis.
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")
    return arr

def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if "rawdata" in lname or "raw" in lname or "data" in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if "eeg" in lname:
        score += 1
    return score

def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({"0", "1", "2"}):
        score += 2
    return score

def validate_liu_source_subject(rawdata, labels, subject_id, path=None):
    """Validate the fixed Liu source MAT layout assumptions."""
    expected_trials = LIU_EXPECTED_TRIALS_PER_SUBJECT
    expected_channels = LIU_EXPECTED_SOURCE_CHANNELS
    expected_samples = LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL

    if rawdata.shape[0] != labels.size:
        raise ValueError(
            f"Subject {subject_id}: labels/trials mismatch. "
            f"rawdata={rawdata.shape}, labels={labels.shape}, path={path}"
        )
    if rawdata.shape[0] != expected_trials:
        print(f"WARNING subject {subject_id}: expected {expected_trials} trials, got {rawdata.shape[0]}")
    if rawdata.shape[1] < len(SOURCE_EEG_CHANNEL_INDICES_30):
        raise ValueError(f"Subject {subject_id}: expected at least 30 EEG-like channels, got {rawdata.shape}")
    if rawdata.shape[1] != expected_channels:
        print(f"WARNING subject {subject_id}: expected {expected_channels} source channels, got {rawdata.shape[1]}")
    if rawdata.shape[2] != expected_samples:
        print(f"WARNING subject {subject_id}: expected {expected_samples} samples/trial, got {rawdata.shape[2]}")

    unique = set(np.unique(labels).astype(int).tolist())
    if not unique.issubset({0, 1, 2}):
        raise ValueError(f"Subject {subject_id}: unexpected labels {sorted(unique)}")

    y0 = labels_to_zero_based(labels)
    counts = np.bincount(y0, minlength=TARGET_N_CLASSES)
    if counts.min() == 0:
        raise ValueError(f"Subject {subject_id}: missing class after zero-based conversion, counts={counts.tolist()}")
    if counts[0] != counts[1]:
        print(f"WARNING subject {subject_id}: class counts are not balanced: {counts.tolist()}")

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)

    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates, label_candidates = [], []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f"mat_structure_failure_{Path(path).stem}.csv"
        preview.to_csv(preview_path, index=False)
        print(f"MAT structure preview for failure saved to: {preview_path}")
        display(preview.head(40))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}")

    _, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    _, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    labels = np.asarray(label_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels=labels).astype(np.float64)

    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label count mismatch in {path}: labels={labels.shape}, rawdata={rawdata.shape}")

    return rawdata, labels.astype(int), raw_name, label_name


## 3.2 EEGNet Preprocessing

In [ ]:
def make_liu_info(sfreq):
    info = mne.create_info(
        ch_names=EEG_CHANNEL_NAMES,
        sfreq=float(sfreq),
        ch_types=["eeg"] * len(EEG_CHANNEL_NAMES), # type: ignore[list-item]
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    unique = set(np.unique(labels).tolist())
    if unique.issubset({1, 2}):
        return labels - 1
    if unique.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(unique)}")

def source_microvolts_to_mne_volts(data):
    """Liu source `.mat` values are treated as microvolts. MNE RawArray expects volts."""
    return np.asarray(data, dtype=np.float64) * 1e-6

def mne_volts_to_microvolts(data):
    return np.asarray(data, dtype=np.float64) * 1e6

def baseline_correct_epochs(X):
    """Subtract each epoch/channel temporal mean. Shape: trials x channels x samples."""
    return X - X.mean(axis=-1, keepdims=True)

def preprocess_subject_eegnet_style(rawdata, labels, subject_id):
    """Apply an EEGNet-oriented preprocessing path to one Liu2024 source subject.

    Path:
        select 29 EEG channels/drop CPz+EOG+marker
        -> convert source microvolts to MNE volts
        -> average reference
        -> bandpass filter
        -> resample
        -> convert back to microvolts
        -> crop a fixed 4-second MI window
        -> optional per-epoch baseline correction

    Train/test standardization is intentionally not done here. It is done inside each CV fold
    using training-fold statistics only.
    """
    if rawdata.ndim != 3:
        raise ValueError(f"Subject {subject_id}: expected 3D rawdata, got {rawdata.shape}")

    n_trials = rawdata.shape[0]
    X_eeg = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)
    X_eeg_volts = source_microvolts_to_mne_volts(X_eeg)

    continuous = X_eeg_volts.transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES), -1)
    info = make_liu_info(LIU_SOURCE_SFREQ)
    raw = mne.io.RawArray(continuous, info, verbose=False)

    if bool(CONFIG.get("apply_average_reference", True)):
        raw.set_eeg_reference("average", projection=False, verbose=False)

    raw.filter(float(CONFIG["bandpass_low"]), float(CONFIG["bandpass_high"]), verbose=False)
    raw.resample(float(CONFIG["sfreq"]), verbose=False)

    data = mne_volts_to_microvolts(raw.get_data())

    expected_samples_per_trial = int(round(rawdata.shape[2] * float(CONFIG["sfreq"]) / float(LIU_SOURCE_SFREQ)))
    total_expected = n_trials * expected_samples_per_trial
    if data.shape[1] != total_expected:
        n_full = data.shape[1] // n_trials
        expected_samples_per_trial = n_full
        data = data[:, :n_trials * expected_samples_per_trial]

    X_rs = data.reshape(len(EEG_CHANNEL_INDICES), n_trials, expected_samples_per_trial).transpose(1, 0, 2)
    if MI_WINDOW_STOP_SAMPLE > X_rs.shape[-1]:
        raise ValueError(
            f"Subject {subject_id}: crop {MI_WINDOW_START_SAMPLE}:{MI_WINDOW_STOP_SAMPLE} "
            f"exceeds trial length {X_rs.shape[-1]} samples after resampling."
        )

    X_win = X_rs[:, :, MI_WINDOW_START_SAMPLE:MI_WINDOW_STOP_SAMPLE]
    if bool(CONFIG.get("baseline_correct_epochs", True)):
        X_win = baseline_correct_epochs(X_win)

    y = labels_to_zero_based(labels)
    return X_win.astype(np.float32), y.astype(np.int64), int(expected_samples_per_trial)


## 3.3 Dataset Classes

In [ ]:
class SubjectArrayDataset(Dataset):
    def __init__(self, X, y, subject_id):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subject_id = str(subject_id)

    def __len__(self):
        return int(len(self.y))

    def __getitem__(self, idx):
        return self.X[idx], int(self.y[idx])


## 3.4 Locate, Load, and Preprocess Source MAT Files

In [ ]:
SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
MAT_FILES = []

files = find_source_mat_files(SOURCE_EXTRACT_DIR)
if files:
    MAT_FILES = files

if not MAT_FILES:
    raise FileNotFoundError("Could not find Liu2024 source .mat files.")

print(f"Source extract dir: {SOURCE_EXTRACT_DIR}")
print(f"Found {len(MAT_FILES)} .mat files")

preview_path = ARTIFACT_DIR / "mat_structure_preview_first_subject.csv"
mat_structure_preview(MAT_FILES[0]).to_csv(preview_path, index=False)
print(f"MAT structure preview saved to: {preview_path}")

subjects = []
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    if CONFIG["subjects_to_use"] is not None and sid not in set(int(s) for s in CONFIG["subjects_to_use"]):
        continue
    if sid in set(int(s) for s in CONFIG["exclude_subjects"]):
        continue
    X_raw, y_raw, raw_field, label_field = load_subject_mat(p)
    validate_liu_source_subject(X_raw, y_raw, sid, path=p)
    subjects.append({
        "subject_id": sid,
        "path": str(p),
        "rawdata_shape": tuple(X_raw.shape),
        "labels_shape": tuple(y_raw.shape),
        "label_counts_raw": np.bincount(y_raw.astype(int), minlength=3).tolist(),
        "raw_field": raw_field,
        "label_field": label_field,
    })

subjects_df = pd.DataFrame(subjects).sort_values("subject_id").reset_index(drop=True)
if subjects_df.empty:
    raise RuntimeError("No subjects loaded.")

SUBJECTS = [int(s) for s in subjects_df["subject_id"].tolist()]
print(f"Subjects loaded: {SUBJECTS}")

subject_inventory_path = ARTIFACT_DIR / "subject_inventory.csv"
subjects_df.to_csv(subject_inventory_path, index=False)
print(f"Subject inventory saved to: {subject_inventory_path}")

EEG_INFO = make_liu_info(CONFIG["sfreq"])
CHS_INFO = EEG_INFO["chs"]
CH_NAMES = list(EEG_CHANNEL_NAMES)

Xs, ys, subject_ids = [], [], []
window_summary_rows = []
for item in subjects_df.to_dict("records"):
    sid = int(item["subject_id"])
    X_raw, y_raw, _, _ = load_subject_mat(Path(item["path"]))
    X_win, y, samples_per_trial = preprocess_subject_eegnet_style(X_raw, y_raw, sid)
    Xs.append(X_win)
    ys.append(y)
    subject_ids.extend([sid] * len(y))
    window_summary_rows.append({
        "subject_id": sid,
        "n_windows": int(len(y)),
        "class_counts": np.bincount(y, minlength=TARGET_N_CLASSES).tolist(),
        "preprocessed_shape": tuple(X_win.shape),
        "resampled_samples_per_trial": int(samples_per_trial),
        "crop_start_sample": int(MI_WINDOW_START_SAMPLE),
        "crop_stop_sample": int(MI_WINDOW_STOP_SAMPLE),
        "target_window_samples": int(WINDOW_SAMPLES),
        "effective_window_duration_s": float(TARGET_TRIAL_DURATION_S),
        "preprocessing_path": "EEGNetv4: average_ref -> bandpass -> resample -> crop -> epoch_baseline",
        "source_values_assumed": "microvolts",
        "model_input_units_before_fold_standardization": "microvolts",
    })

X_ALL = np.concatenate(Xs, axis=0)
Y_ALL = np.concatenate(ys, axis=0)
SUBJECT_ID_ALL = np.asarray(subject_ids)
print(f"X_ALL shape: {X_ALL.shape} | Y_ALL counts: {np.bincount(Y_ALL).tolist()}")

window_summary_df = pd.DataFrame(window_summary_rows).sort_values("subject_id")
window_summary_path = ARTIFACT_DIR / "window_counts_by_subject.csv"
window_summary_df.to_csv(window_summary_path, index=False)
print(f"Window summary saved to: {window_summary_path}")
display(window_summary_df.head())

def _sort_subject_key(x):
    sx = str(x)
    return int(sx) if sx.isdigit() else sx

SUBJECT_WINDOWS = {}
for sid in sorted(np.unique(SUBJECT_ID_ALL), key=_sort_subject_key):
    idx = np.where(SUBJECT_ID_ALL == sid)[0]
    SUBJECT_WINDOWS[str(sid)] = SubjectArrayDataset(X_ALL[idx], Y_ALL[idx], subject_id=sid)


# 4. Model

In [ ]:
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total_params": int(total), "trainable_params": int(trainable)}

def build_eegnet_model():
    """Build Braindecode EEGNetv4 with compatibility for newer and older Braindecode APIs."""
    modern_kwargs = dict(
        n_chans=len(CH_NAMES),
        n_outputs=TARGET_N_CLASSES,
        n_times=WINDOW_SAMPLES,
        final_conv_length="auto",
        pool_mode=CONFIG["eegnet_pool_mode"],
        F1=int(CONFIG["eegnet_F1"]),
        D=int(CONFIG["eegnet_D"]),
        F2=int(CONFIG["eegnet_F2"]),
        kernel_length=int(CONFIG["eegnet_kernel_length"]),
        drop_prob=float(CONFIG["eegnet_drop_prob"]),
    )
    try:
        model = EEGNetv4(**modern_kwargs)
        api_used = "modern:n_chans/n_outputs/n_times"
        kwargs_used = modern_kwargs
    except TypeError:
        legacy_kwargs = dict(
            in_chans=len(CH_NAMES),
            n_classes=TARGET_N_CLASSES,
            input_window_samples=WINDOW_SAMPLES,
            final_conv_length="auto",
            pool_mode=CONFIG["eegnet_pool_mode"],
            F1=int(CONFIG["eegnet_F1"]),
            D=int(CONFIG["eegnet_D"]),
            F2=int(CONFIG["eegnet_F2"]),
            kernel_length=int(CONFIG["eegnet_kernel_length"]),
            drop_prob=float(CONFIG["eegnet_drop_prob"]),
        )
        model = EEGNetv4(**legacy_kwargs)
        api_used = "legacy:in_chans/n_classes/input_window_samples"
        kwargs_used = legacy_kwargs

    return model, {"api_used": api_used, "kwargs": kwargs_used, **count_parameters(model)}

model_preview, model_info = build_eegnet_model()
print("EEGNetv4 model info:")
print(json.dumps(model_info, indent=2, default=str))

with torch.no_grad():
    dummy = torch.zeros(2, len(CH_NAMES), WINDOW_SAMPLES, dtype=torch.float32)
    out = model_preview(dummy)
    if isinstance(out, tuple):
        out_shape = [list(o.shape) for o in out if hasattr(o, "shape")]
    else:
        out_shape = list(out.shape)
print(f"Forward check output shape: {out_shape}")


# 5. Training

## 5.1 Fold Standardization and Metrics

In [ ]:
def get_targets(dataset):
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

def standardize_by_train_fold(X_train, X_test):
    """Standardize using training-fold statistics only to avoid test leakage."""
    if not bool(CONFIG.get("standardize_using_train_fold", True)):
        return X_train.astype(np.float32), X_test.astype(np.float32), None

    mode = CONFIG.get("standardize_mode", "channel")
    if mode == "channel":
        mean = X_train.mean(axis=(0, 2), keepdims=True)
        std = X_train.std(axis=(0, 2), keepdims=True)
    elif mode == "global":
        mean = X_train.mean(keepdims=True)
        std = X_train.std(keepdims=True)
    else:
        raise ValueError("CONFIG['standardize_mode'] must be 'channel' or 'global'.")

    std = np.maximum(std, 1e-6)
    X_train_z = (X_train - mean) / std
    X_test_z = (X_test - mean) / std

    stats = {
        "mode": mode,
        "mean_shape": list(mean.shape),
        "std_shape": list(std.shape),
        "mean_abs_mean": float(np.mean(np.abs(mean))),
        "std_mean": float(np.mean(std)),
        "std_min": float(np.min(std)),
        "std_max": float(np.max(std)),
    }
    return X_train_z.astype(np.float32), X_test_z.astype(np.float32), stats

def make_fold_datasets(subject_dataset, train_idx, test_idx):
    X_train = subject_dataset.X[np.asarray(train_idx, dtype=int)]
    y_train = subject_dataset.y[np.asarray(train_idx, dtype=int)]
    X_test = subject_dataset.X[np.asarray(test_idx, dtype=int)]
    y_test = subject_dataset.y[np.asarray(test_idx, dtype=int)]

    X_train, X_test, standardization_stats = standardize_by_train_fold(X_train, X_test)

    return (
        SubjectArrayDataset(X_train, y_train, subject_id=subject_dataset.subject_id),
        SubjectArrayDataset(X_test, y_test, subject_id=subject_dataset.subject_id),
        standardization_stats,
    )

def _json_safe_float(value, decimals=8):
    if value is None:
        return None
    value = float(value)
    if not np.isfinite(value):
        return None
    return round(value, decimals)

def _json_safe_float_list(values, decimals=8):
    return [_json_safe_float(v, decimals=decimals) for v in values]

def compute_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_pred = np.asarray(y_pred).astype(int).reshape(-1)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }

def compute_collapse_diagnostics(y_pred, n_classes):
    pred_hist = np.bincount(np.asarray(y_pred, dtype=int), minlength=n_classes)
    n_pred = int(pred_hist.sum())
    collapse_ratio = float(pred_hist.max() / n_pred) if n_pred else 0.0
    threshold = float(CONFIG.get("collapse_threshold", 0.90))
    return {
        "prediction_histogram": pred_hist.tolist(),
        "collapse_ratio": _json_safe_float(collapse_ratio),
        "collapse_threshold": threshold,
        "collapse_flag": bool(collapse_ratio >= threshold),
        "majority_predicted_class": int(pred_hist.argmax()) if n_pred else None,
    }

def compute_prediction_probability_diagnostics(clf, test_set, y_pred, n_classes):
    if not bool(CONFIG.get("log_probability_diagnostics", True)):
        return None
    try:
        probs = np.asarray(clf.predict_proba(test_set), dtype=float)
    except Exception as exc:
        return {"available": False, "reason": f"predict_proba failed: {exc}"}

    probs = np.squeeze(probs)
    if probs.ndim != 2 or probs.shape[0] != len(test_set):
        return {"available": False, "reason": f"Unexpected probability shape: {list(probs.shape)}"}
    if probs.shape[1] != n_classes:
        return {"available": False, "reason": f"Unexpected class dimension: {list(probs.shape)}"}
    if not np.isfinite(probs).all():
        return {"available": False, "reason": "Non-finite probabilities."}

    row_sums = probs.sum(axis=1, keepdims=True)
    if np.any(probs < 0) or not np.allclose(row_sums, 1.0, atol=1e-3):
        exp_probs = np.exp(probs - probs.max(axis=1, keepdims=True))
        probs = exp_probs / np.maximum(exp_probs.sum(axis=1, keepdims=True), 1e-12)

    eps = 1e-12
    confidence = probs.max(axis=1)
    entropy = -np.sum(probs * np.log(probs + eps), axis=1)
    normalized_entropy = entropy / np.log(max(probs.shape[1], 2))
    predicted_class_probability = probs[np.arange(len(probs)), np.asarray(y_pred, dtype=int)]

    return {
        "available": True,
        "probability_shape": list(probs.shape),
        "mean_probability_by_class": _json_safe_float_list(probs.mean(axis=0)),
        "std_probability_by_class": _json_safe_float_list(probs.std(axis=0)),
        "mean_confidence": _json_safe_float(confidence.mean()),
        "std_confidence": _json_safe_float(confidence.std()),
        "mean_prediction_entropy": _json_safe_float(entropy.mean()),
        "mean_normalized_prediction_entropy": _json_safe_float(normalized_entropy.mean()),
        "mean_predicted_class_probability": _json_safe_float(predicted_class_probability.mean()),
    }


## 5.2 Build Classifier

In [ ]:
def make_train_split():
    val_split = CONFIG["val_split"]
    if val_split is None or float(val_split) <= 0.0:
        return None
    return ValidSplit(cv=float(val_split), stratified=True, random_state=BASE_SEED if BASE_SEED is not None else 12)

def make_callbacks():
    train_split = make_train_split()
    patience = CONFIG["early_stopping_patience"]
    if train_split is None or patience is None or int(patience) <= 0:
        return []
    return [
        (
            "early_stopping",
            EarlyStopping(
                monitor="valid_loss",
                patience=int(patience),
                lower_is_better=True,
                load_best=True,
            ),
        )
    ]

def build_classifier(model, fold_seed=None):
    train_generator = None
    if fold_seed is not None:
        train_generator = torch.Generator()
        train_generator.manual_seed(int(fold_seed))

    clf_kwargs = {
        "criterion": torch.nn.CrossEntropyLoss,
        "optimizer": torch.optim.AdamW,
        "optimizer__lr": float(CONFIG["learning_rate"]),
        "optimizer__weight_decay": float(CONFIG["weight_decay"]),
        "batch_size": int(CONFIG["batch_size"]),
        "max_epochs": int(CONFIG["n_epochs"]),
        "device": DEVICE,
        "callbacks": make_callbacks(),
        "train_split": make_train_split(),
        "classes": range(TARGET_N_CLASSES),
        "iterator_train__shuffle": True,
        "iterator_train__num_workers": 0,
        "iterator_valid__num_workers": 0,
        "verbose": 0,
    }
    if train_generator is not None:
        clf_kwargs["iterator_train__generator"] = train_generator

    return EEGClassifier(model, **clf_kwargs)

def history_to_rows(history, subject_id, fold_id):
    rows = []
    for r in history:
        row = {"subject_id": str(subject_id), "fold_id": int(fold_id)}
        for key in ["epoch", "train_loss", "valid_loss", "dur"]:
            if key in r and r[key] is not None:
                try:
                    row[key] = float(r[key]) if key != "epoch" else int(r[key])
                except Exception:
                    row[key] = str(r[key])
        rows.append(row)
    return rows


## 5.3 Subject Cross-Validation Runner

In [ ]:
TRAINING_HISTORY_ROWS = []
MODEL_INFO_BY_FOLD = []

def run_training_and_eval(train_set, test_set, fold_id, fold_label, standardization_stats=None, n_total_folds=None):
    if CONFIG["set_seed"]:
        fold_seed = int(BASE_SEED) + int(fold_id) if BASE_SEED is not None else None
        seed_everything(fold_seed)
    else:
        fold_seed = None

    y_train = get_targets(train_set)
    y_test = get_targets(test_set)
    train_counts = np.bincount(y_train, minlength=TARGET_N_CLASSES)
    test_counts = np.bincount(y_test, minlength=TARGET_N_CLASSES)

    model, model_info = build_eegnet_model()
    MODEL_INFO_BY_FOLD.append({"fold_id": int(fold_id), "fold_label": str(fold_label), **model_info})

    fold_tag = f"/{n_total_folds}" if n_total_folds is not None else ""
    print(f"\nFold {fold_id}{fold_tag} | {fold_label}")
    print(f"    Train: {len(train_set)} | counts={train_counts.tolist()}")
    print(f"    Test:  {len(test_set)} | counts={test_counts.tolist()}")
    print(f"    Model: {CONFIG['model_name']} | params={model_info['trainable_params']:,}")
    print(f"    Standardization: {standardization_stats}")

    clf = build_classifier(model, fold_seed=fold_seed)
    clf.fit(train_set, y=y_train)

    y_pred = np.asarray(clf.predict(test_set), dtype=int).reshape(-1)
    metrics = compute_classification_metrics(y_test, y_pred)
    collapse = compute_collapse_diagnostics(y_pred, TARGET_N_CLASSES)
    probability_diagnostics = compute_prediction_probability_diagnostics(clf, test_set, y_pred, TARGET_N_CLASSES)

    history_rows = list(clf.history)
    TRAINING_HISTORY_ROWS.extend(history_to_rows(history_rows, subject_id=str(fold_label).split("=")[-1], fold_id=fold_id))

    valid_loss_curve = [
        (int(r["epoch"]), float(r["valid_loss"]))
        for r in history_rows
        if "valid_loss" in r and r["valid_loss"] is not None
    ]
    best_epoch, best_valid_loss = (
        min(valid_loss_curve, key=lambda x: x[1]) if valid_loss_curve else (None, None)
    )
    stopped_epoch = int(history_rows[-1]["epoch"]) if history_rows else 0

    cm = confusion_matrix(y_test, y_pred, labels=np.arange(TARGET_N_CLASSES)).tolist()

    print(
        f"    Result | best_epoch={best_epoch} | stop={stopped_epoch} | "
        f"acc={metrics['accuracy']:.4f} | bal_acc={metrics['balanced_accuracy']:.4f} | "
        f"pred_hist={collapse['prediction_histogram']} | collapse={collapse['collapse_flag']}"
    )

    return {
        "fold_id": int(fold_id),
        "fold_label": str(fold_label),
        "model_name": CONFIG["model_name"],
        "n_train": int(len(train_set)),
        "n_test": int(len(test_set)),
        "train_class_counts": train_counts.tolist(),
        "test_class_counts": test_counts.tolist(),
        "model_info": model_info,
        "fold_seed": fold_seed,
        "standardization_stats": standardization_stats,
        "best_epoch": best_epoch,
        "stopped_epoch": int(stopped_epoch),
        "epochs_ran": int(len(history_rows)),
        "best_valid_loss": best_valid_loss,
        "tested_checkpoint": "best_valid_loss_via_skorch_earlystopping_load_best" if valid_loss_curve else "final_model_no_validation_history",
        "accuracy": metrics["accuracy"],
        "balanced_accuracy": metrics["balanced_accuracy"],
        "confusion_matrix": cm,
        "y_true": y_test.astype(int).tolist(),
        "y_pred": y_pred.astype(int).tolist(),
        "prediction_histogram": collapse["prediction_histogram"],
        "collapse_diagnostics": collapse,
        "probability_diagnostics": probability_diagnostics,
    }

def make_fold_splits(y, n_folds, n_classes):
    counts = np.bincount(y, minlength=n_classes)
    if counts.min() < n_folds:
        raise ValueError(f"Cannot use {n_folds} folds with class counts={counts.tolist()}.")

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=BASE_SEED if BASE_SEED is not None else 12)
    indices = np.arange(len(y))
    folds = []
    for fold_id, (train_idx, test_idx) in enumerate(skf.split(indices, y), start=1):
        if CONFIG.get("assert_balanced_folds", True) and np.all(counts % n_folds == 0):
            expected_test = (counts // n_folds).astype(int)
            expected_train = (counts - expected_test).astype(int)
            train_counts = np.bincount(y[train_idx], minlength=n_classes)
            test_counts = np.bincount(y[test_idx], minlength=n_classes)
            assert np.array_equal(train_counts, expected_train), (
                f"Unexpected train class counts for fold {fold_id}: "
                f"{train_counts.tolist()} != {expected_train.tolist()}"
            )
            assert np.array_equal(test_counts, expected_test), (
                f"Unexpected test class counts for fold {fold_id}: "
                f"{test_counts.tolist()} != {expected_test.tolist()}"
            )
        folds.append({"fold_id": fold_id, "idx_train": train_idx, "idx_test": test_idx})
    return folds

def run_subject_cv(subject_id, subject_dataset, n_classes, cv_folds):
    y = get_targets(subject_dataset)
    counts = np.bincount(y, minlength=n_classes)
    print(f"\nSubject {subject_id}: {len(subject_dataset)} windows | class_counts={counts.tolist()}")
    folds = make_fold_splits(y, n_folds=cv_folds, n_classes=n_classes)
    results = []
    for fold in folds:
        train_set, test_set, standardization_stats = make_fold_datasets(
            subject_dataset,
            fold["idx_train"],
            fold["idx_test"],
        )
        result = run_training_and_eval(
            train_set,
            test_set,
            fold["fold_id"],
            f"subject={subject_id}",
            standardization_stats=standardization_stats,
            n_total_folds=cv_folds,
        )
        result["subject_id"] = str(subject_id)
        results.append(result)

    acc_values = [r["accuracy"] for r in results if r["accuracy"] is not None]
    bal_values = [r["balanced_accuracy"] for r in results if r["balanced_accuracy"] is not None]
    print(
        f"  Subject {subject_id}: acc={np.mean(acc_values):.4f}±{np.std(acc_values):.4f}  "
        f"bal_acc={np.mean(bal_values):.4f}±{np.std(bal_values):.4f}"
    )
    return results


## 5.4 Run All Subjects

In [ ]:
print("=" * 70)
print("STARTING WITHIN-SUBJECT EEGNETV4 CROSS-VALIDATION")
print("=" * 70)
print(f"Subjects:      {sorted(SUBJECT_WINDOWS.keys(), key=_sort_subject_key)}")
print(f"Model:         {CONFIG['model_name']}")
print(f"Optimizer:     AdamW | lr={CONFIG['learning_rate']} | weight_decay={CONFIG['weight_decay']}")
print(f"CV folds:      {CONFIG['cv_folds']}")
print(f"Val split:     {CONFIG['val_split']}")
print(f"Max epochs:    {CONFIG['n_epochs']}")
print(f"Early stop:    {CONFIG['early_stopping_patience']}")
print(f"Device:        {DEVICE}")
print("=" * 70)

FOLD_RESULTS = []
for sid in sorted(SUBJECT_WINDOWS.keys(), key=_sort_subject_key):
    FOLD_RESULTS.extend(run_subject_cv(sid, SUBJECT_WINDOWS[sid], TARGET_N_CLASSES, CONFIG["cv_folds"]))
print(f"\nTotal folds completed: {len(FOLD_RESULTS)}")


# 6. Results

## 6.1 Aggregate Metrics

In [ ]:
def aggregate_results(fold_results):
    grouped = {}
    for result in fold_results:
        sid = result.get("subject_id", "global")
        grouped.setdefault(sid, {"accuracies": [], "balanced_accuracies": []})
        grouped[sid]["accuracies"].append(result.get("accuracy"))
        grouped[sid]["balanced_accuracies"].append(result.get("balanced_accuracy"))

    for sid, metrics in grouped.items():
        acc_values = [v for v in metrics["accuracies"] if v is not None]
        bal_values = [v for v in metrics["balanced_accuracies"] if v is not None]
        metrics["mean_accuracy"] = float(np.mean(acc_values)) if acc_values else None
        metrics["std_accuracy"] = float(np.std(acc_values)) if acc_values else None
        metrics["mean_balanced_accuracy"] = float(np.mean(bal_values)) if bal_values else None
        metrics["std_balanced_accuracy"] = float(np.std(bal_values)) if bal_values else None

    all_accs = [r["accuracy"] for r in fold_results if r.get("accuracy") is not None]
    all_bals = [r["balanced_accuracy"] for r in fold_results if r.get("balanced_accuracy") is not None]
    all_true = []
    all_pred = []
    for r in fold_results:
        all_true.extend(r.get("y_true", []))
        all_pred.extend(r.get("y_pred", []))

    cm = confusion_matrix(all_true, all_pred, labels=np.arange(TARGET_N_CLASSES)).tolist() if all_true else None
    pred_hist = np.bincount(np.asarray(all_pred, dtype=int), minlength=TARGET_N_CLASSES).tolist() if all_pred else None

    global_metrics = {
        "mean_accuracy": float(np.mean(all_accs)) if all_accs else None,
        "std_accuracy": float(np.std(all_accs)) if all_accs else None,
        "mean_balanced_accuracy": float(np.mean(all_bals)) if all_bals else None,
        "std_balanced_accuracy": float(np.std(all_bals)) if all_bals else None,
        "n_folds": int(len(fold_results)),
        "n_predictions": int(len(all_true)),
        "confusion_matrix": cm,
        "prediction_histogram": pred_hist,
    }
    return grouped, global_metrics

SUBJECT_METRICS, GLOBAL_METRICS = aggregate_results(FOLD_RESULTS)

print("Global metrics:")
print(json.dumps(GLOBAL_METRICS, indent=2))

subject_summary = pd.DataFrame([
    {
        "subject_id": sid,
        "mean_accuracy": vals["mean_accuracy"],
        "std_accuracy": vals["std_accuracy"],
        "mean_balanced_accuracy": vals["mean_balanced_accuracy"],
        "std_balanced_accuracy": vals["std_balanced_accuracy"],
        "n_folds": len(vals["accuracies"]),
    }
    for sid, vals in SUBJECT_METRICS.items()
]).sort_values("mean_balanced_accuracy", ascending=False)

display(subject_summary.head(20))


## 6.2 Performance Tables and Plots

In [ ]:
def _safe_percent(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan
    return float(value) * 100.0

def _jsonable(value):
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value

def build_performance_tables(fold_results):
    fold_rows = []
    for result in fold_results:
        prob = result.get("probability_diagnostics") or {}
        collapse = result.get("collapse_diagnostics") or {}
        fold_rows.append({
            "method_name": f"{CONFIG['model_name']}__eegnet_preprocess",
            "model_name": CONFIG["model_name"],
            "subject_id": str(result.get("subject_id")),
            "fold_id": int(result.get("fold_id")),
            "accuracy": result.get("accuracy"),
            "accuracy_percent": _safe_percent(result.get("accuracy")),
            "balanced_accuracy": result.get("balanced_accuracy"),
            "balanced_accuracy_percent": _safe_percent(result.get("balanced_accuracy")),
            "n_train": result.get("n_train"),
            "n_test": result.get("n_test"),
            "train_class_counts": json.dumps(result.get("train_class_counts")),
            "test_class_counts": json.dumps(result.get("test_class_counts")),
            "prediction_histogram": json.dumps(result.get("prediction_histogram")),
            "collapse_ratio": collapse.get("collapse_ratio"),
            "collapse_flag": collapse.get("collapse_flag"),
            "mean_confidence": prob.get("mean_confidence") if prob.get("available") else None,
            "mean_normalized_prediction_entropy": prob.get("mean_normalized_prediction_entropy") if prob.get("available") else None,
            "best_epoch": result.get("best_epoch"),
            "stopped_epoch": result.get("stopped_epoch"),
            "epochs_ran": result.get("epochs_ran"),
            "best_valid_loss": result.get("best_valid_loss"),
            "fold_seed": result.get("fold_seed"),
        })
    fold_df = pd.DataFrame(fold_rows)

    subject_df = (
        fold_df
        .groupby(["method_name", "model_name", "subject_id"], as_index=False)
        .agg(
            mean_accuracy=("accuracy", "mean"),
            std_accuracy=("accuracy", "std"),
            mean_accuracy_percent=("accuracy_percent", "mean"),
            std_accuracy_percent=("accuracy_percent", "std"),
            mean_balanced_accuracy=("balanced_accuracy", "mean"),
            std_balanced_accuracy=("balanced_accuracy", "std"),
            mean_balanced_accuracy_percent=("balanced_accuracy_percent", "mean"),
            std_balanced_accuracy_percent=("balanced_accuracy_percent", "std"),
            collapse_rate=("collapse_flag", "mean"),
            mean_confidence=("mean_confidence", "mean"),
            mean_normalized_prediction_entropy=("mean_normalized_prediction_entropy", "mean"),
            n_folds=("fold_id", "count"),
        )
        .sort_values("mean_balanced_accuracy", ascending=False)
    )

    global_df = pd.DataFrame([{
        "method_name": f"{CONFIG['model_name']}__eegnet_preprocess",
        "model_name": CONFIG["model_name"],
        "mean_accuracy": GLOBAL_METRICS["mean_accuracy"],
        "std_accuracy": GLOBAL_METRICS["std_accuracy"],
        "mean_accuracy_percent": _safe_percent(GLOBAL_METRICS["mean_accuracy"]),
        "std_accuracy_percent": _safe_percent(GLOBAL_METRICS["std_accuracy"]),
        "mean_balanced_accuracy": GLOBAL_METRICS["mean_balanced_accuracy"],
        "std_balanced_accuracy": GLOBAL_METRICS["std_balanced_accuracy"],
        "mean_balanced_accuracy_percent": _safe_percent(GLOBAL_METRICS["mean_balanced_accuracy"]),
        "std_balanced_accuracy_percent": _safe_percent(GLOBAL_METRICS["std_balanced_accuracy"]),
        "n_folds": GLOBAL_METRICS["n_folds"],
        "n_predictions": GLOBAL_METRICS["n_predictions"],
        "confusion_matrix": json.dumps(GLOBAL_METRICS["confusion_matrix"]),
        "prediction_histogram": json.dumps(GLOBAL_METRICS["prediction_histogram"]),
    }])
    return fold_df, subject_df, global_df

fold_level_df, subject_level_df, global_method_df = build_performance_tables(FOLD_RESULTS)

fold_level_results_path = ARTIFACT_DIR / "fold_level_results.csv"
subject_level_summary_path = ARTIFACT_DIR / "subject_level_summary.csv"
global_method_comparison_path = ARTIFACT_DIR / "global_method_comparison.csv"
training_history_path = ARTIFACT_DIR / "training_history_long.csv"

fold_level_df.to_csv(fold_level_results_path, index=False)
subject_level_df.to_csv(subject_level_summary_path, index=False)
global_method_df.to_csv(global_method_comparison_path, index=False)
pd.DataFrame(TRAINING_HISTORY_ROWS).to_csv(training_history_path, index=False)

print("Saved performance tables:")
print(f"  Fold-level:       {fold_level_results_path}")
print(f"  Subject-level:    {subject_level_summary_path}")
print(f"  Global summary:   {global_method_comparison_path}")
print(f"  Training history: {training_history_path}")

display(global_method_df)
display(subject_level_df.head(15))

# Subject-level performance plot.
plot_df = subject_level_df.copy()
plot_df["subject_id_int"] = plot_df["subject_id"].astype(int)
plot_df = plot_df.sort_values("subject_id_int")

fig, ax = plt.subplots(figsize=(14, 4.8))
ax.bar(plot_df["subject_id"], plot_df["mean_balanced_accuracy_percent"], yerr=plot_df["std_balanced_accuracy_percent"].fillna(0.0), capsize=2)
ax.axhline(50.0, linestyle="--", linewidth=1)
ax.set_xlabel("Subject")
ax.set_ylabel("Balanced accuracy (%)")
ax.set_title("EEGNetv4 within-subject balanced accuracy by subject")
ax.tick_params(axis="x", rotation=90)
fig.tight_layout()
subject_performance_plot_path = ARTIFACT_DIR / "subject_balanced_accuracy.png"
fig.savefig(subject_performance_plot_path, dpi=160)
plt.show()

# Global summary plot.
fig, ax = plt.subplots(figsize=(5.5, 4.5))
mean_val = float(global_method_df.loc[0, "mean_balanced_accuracy_percent"])
std_val = float(global_method_df.loc[0, "std_balanced_accuracy_percent"])
ax.bar(["EEGNetv4"], [mean_val], yerr=[std_val], capsize=6)
ax.axhline(50.0, linestyle="--", linewidth=1)
ax.set_ylabel("Balanced accuracy (%)")
ax.set_title("Global EEGNetv4 baseline")
fig.tight_layout()
global_performance_plot_path = ARTIFACT_DIR / "global_balanced_accuracy.png"
fig.savefig(global_performance_plot_path, dpi=160)
plt.show()

# Aggregated confusion matrix.
cm = np.asarray(GLOBAL_METRICS["confusion_matrix"], dtype=int)
fig, ax = plt.subplots(figsize=(4.8, 4.2))
im = ax.imshow(cm)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred 0", "Pred 1"])
ax.set_yticklabels(["True 0", "True 1"])
ax.set_title("Aggregated confusion matrix")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
confusion_performance_plot_path = ARTIFACT_DIR / "aggregated_confusion_matrix.png"
fig.savefig(confusion_performance_plot_path, dpi=160)
plt.show()

confusion_metrics_path = ARTIFACT_DIR / "eegnet_confusion_metrics.csv"
confusion_rows = []
for i in range(TARGET_N_CLASSES):
    tp = int(cm[i, i])
    total_true = int(cm[i, :].sum())
    recall = tp / total_true if total_true else np.nan
    confusion_rows.append({"class_id": i, "true_total": total_true, "correct": tp, "recall": recall})
pd.DataFrame(confusion_rows).to_csv(confusion_metrics_path, index=False)
print(f"Confusion metrics saved: {confusion_metrics_path}")


## 6.3 Experiment Summary

In [ ]:
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Run ID:                 {RUN_ID}")
print(f"Subjects:               {len(SUBJECTS)}")
print(f"Model:                  {CONFIG['model_name']}")
print(f"Window:                 {TARGET_TRIAL_DURATION_S:.3f}s / {WINDOW_SAMPLES} samples")
print(f"Preprocessing:          avg-ref={CONFIG['apply_average_reference']} | bandpass={CONFIG['bandpass_low']}-{CONFIG['bandpass_high']} Hz | sfreq={CONFIG['sfreq']} Hz")
print(f"Fold standardization:   {CONFIG['standardize_using_train_fold']} ({CONFIG['standardize_mode']})")
print(f"Artifacts:              {ARTIFACT_DIR}")
print(f"Mean Accuracy:          {GLOBAL_METRICS['mean_accuracy']:.4f} ± {GLOBAL_METRICS['std_accuracy']:.4f}")
print(f"Mean Balanced Accuracy: {GLOBAL_METRICS['mean_balanced_accuracy']:.4f} ± {GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
print(f"Confusion matrix:       {GLOBAL_METRICS['confusion_matrix']}")
print("=" * 70)


## 6.4 Save Artifacts

In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, "w") as f:
    json.dump(FOLD_RESULTS, f, indent=2)

subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
with open(subject_metrics_path, "w") as f:
    json.dump(SUBJECT_METRICS, f, indent=2)

global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
with open(global_metrics_path, "w") as f:
    json.dump(GLOBAL_METRICS, f, indent=2)

model_info_path = ARTIFACT_DIR / "model_info_by_fold.json"
with open(model_info_path, "w") as f:
    json.dump(MODEL_INFO_BY_FOLD, f, indent=2, default=str)

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "excluded_subjects": list(CONFIG["exclude_subjects"]),
    "subjects": [str(s) for s in SUBJECTS],
    "source_sfreq": LIU_SOURCE_SFREQ,
    "target_sfreq": CONFIG["sfreq"],
    "bandpass_low": CONFIG["bandpass_low"],
    "bandpass_high": CONFIG["bandpass_high"],
    "preprocessing_order": [
        "select 29 EEG channels; drop CPz source reference, EOG, and marker",
        "treat Liu source values as microvolts and convert to MNE volts",
        "concatenate source trials per subject into MNE RawArray",
        "average reference if enabled",
        "bandpass filter",
        "resample to target sfreq",
        "scale volts to microvolts",
        "reshape back to trials",
        "crop fixed EEGNet MI window",
        "optional per-epoch/channel baseline correction",
        "within each CV fold, standardize train/test using training-fold statistics only",
    ],
    "mi_window_start_s": CONFIG["mi_window_start_s"],
    "target_window_duration_s": TARGET_TRIAL_DURATION_S,
    "window_samples": WINDOW_SAMPLES,
    "mi_window_start_sample": MI_WINDOW_START_SAMPLE,
    "mi_window_stop_sample": MI_WINDOW_STOP_SAMPLE,
    "expected_trials_per_subject": LIU_EXPECTED_TRIALS_PER_SUBJECT,
    "expected_source_channels": LIU_EXPECTED_SOURCE_CHANNELS,
    "expected_source_samples_per_trial": LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL,
    "source_values_assumed": "microvolts",
    "final_model_units_before_fold_standardization": "microvolts",
    "subject_inventory_path": str(subject_inventory_path),
    "window_summary_path": str(window_summary_path),
    "channel_names": list(CH_NAMES),
    "n_channels": len(CH_NAMES),
    "model_name": CONFIG["model_name"],
    "optimizer": "AdamW",
    "learning_rate": CONFIG["learning_rate"],
    "weight_decay": CONFIG["weight_decay"],
    "seed": BASE_SEED,
    "cv_folds": CONFIG["cv_folds"],
    "val_split": CONFIG["val_split"],
    "global_metrics": GLOBAL_METRICS,
    "performance_artifacts": {
        "fold_level_results": str(fold_level_results_path) if "fold_level_results_path" in globals() else None,
        "subject_level_summary": str(subject_level_summary_path) if "subject_level_summary_path" in globals() else None,
        "global_method_comparison": str(global_method_comparison_path) if "global_method_comparison_path" in globals() else None,
        "training_history": str(training_history_path) if "training_history_path" in globals() else None,
        "subject_performance_plot": str(subject_performance_plot_path) if "subject_performance_plot_path" in globals() else None,
        "global_performance_plot": str(global_performance_plot_path) if "global_performance_plot_path" in globals() else None,
        "aggregated_confusion_plot": str(confusion_performance_plot_path) if "confusion_performance_plot_path" in globals() else None,
        "confusion_metrics": str(confusion_metrics_path) if "confusion_metrics_path" in globals() else None,
    },
    "collapse_threshold": float(CONFIG.get("collapse_threshold", 0.90)),
    "log_probability_diagnostics": bool(CONFIG.get("log_probability_diagnostics", True)),
}

run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f:
    json.dump(run_metadata, f, indent=2)

print(f"CV results saved to:      {cv_results_path}")
print(f"Subject metrics saved to: {subject_metrics_path}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Model info saved to:      {model_info_path}")
if "subject_performance_plot_path" in globals():
    print(f"Subject performance plot: {subject_performance_plot_path}")
if "global_performance_plot_path" in globals():
    print(f"Global performance plot:  {global_performance_plot_path}")
if "confusion_performance_plot_path" in globals():
    print(f"Confusion matrix plot:    {confusion_performance_plot_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")

try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass
